RITHESH P - 24BAD098

In [3]:
# EXPT NO: 10
# ROLL NO: 24BAD098
# BUILDING A SIMPLE LLM CHATBOT


!pip install -q -U gradio transformers accelerate


In [4]:

# EXPERIMENT NO: 10
# ROLL NO: 24BAD098

import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded successfully!")
print("Roll Number: 24BAD098")
print("Device:", device)
print("Gradio version:", gr.__version__)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Roll Number: 24BAD098
Device: cuda
Gradio version: 6.30.0


In [5]:
# PART A: SIMPLE PROMPT TEST

prompt = "Hello, how are you?"

input_ids = tokenizer.encode(
    prompt + tokenizer.eos_token,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )

response = tokenizer.decode(
    output_ids[0][input_ids.shape[-1]:],
    skip_special_tokens=True
)

print("User Prompt:", prompt)
print("Chatbot Response:", response)

User Prompt: Hello, how are you?
Chatbot Response: 
A. How are you?
B. Fine, thank you.
C. I'm fine, too.
D. What's up?
Answer:

C

Which of the following statements about the characteristics of a company is incorrect?
A. A company


In [10]:

# EXPERIMENT NO: 10
# ROLL NO: 24BAD098

def generate_response(user_message, history=None):

    if not user_message or not user_message.strip():
        return "Please enter a message."

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant. "
                "Answer clearly in simple English. "
                "Stay relevant to the user's question."
            )
        }
    ]

    if history:
        for item in history[-6:]:
            if not isinstance(item, dict):
                continue

            role = item.get("role")
            content = item.get("content")

            if role in ("user", "assistant") and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })

    messages.append({
        "role": "user",
        "content": user_message.strip()
    })

    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
           outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

        input_length = inputs["input_ids"].shape[-1]
        new_tokens = outputs[0][input_length:]

        answer = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        return answer or "Please try another question."

    except Exception as error:
        print("Generation error:", repr(error))
        return "Sorry, I could not generate a response. Please try again."


In [11]:
# ROLL NO: 24BAD098
# PART B: TESTING WITH FIVE PROMPTS

test_prompts = [
    "Hello!",
    "How are you?",
    "What is artificial intelligence?",
    "What is machine learning?",
    "Tell me a short joke."
]

for i, prompt in enumerate(test_prompts, start=1):
    print("\nTest Case:", i)
    print("Prompt:", prompt)
    print("Response:", generate_response(prompt))
    print("-" * 50)


Test Case: 1
Prompt: Hello!
Response: Hello! How can I help you today? Is there something specific you'd like to know or discuss? I'm here to assist with any questions you might have.
--------------------------------------------------

Test Case: 2
Prompt: How are you?
Response: I'm just a computer program, so I don't have feelings or emotions like humans do. I exist solely to process information and answer questions based on that input. How can I help you today?
--------------------------------------------------

Test Case: 3
Prompt: What is artificial intelligence?
Response: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and act like humans. It involves creating software, hardware, or both that can perform tasks that typically require human intelligence, such as learning from data, understanding language, recognizing patterns, and making decisions based on those patterns.
Key aspects of AI include:
1. Machine Lear

In [12]:

# EXPERIMENT NO: 10
# ROLL NO: 24BAD098
# LLM CHATBOT - ERROR-FREE INTERFACE

import gradio as gr
import html

def chat_function(message, history):
    history = history or []

    if not message or not message.strip():
        transcript = ""
        for item in history:
            transcript += f"**{item['role'].title()}:** {html.escape(item['content'])}\n\n"
        return transcript, history, ""

    user_message = message.strip()

    try:
        response = generate_response(user_message, history)
        if not isinstance(response, str):
            response = str(response)
    except Exception as e:
        print("Chatbot error:", repr(e))
        response = "Sorry, I could not generate a response. Please try again."

    updated_history = history + [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": response}
    ]

    transcript = ""
    for item in updated_history:
        role = item["role"].title()
        content = html.escape(item["content"])
        transcript += f"**{role}:** {content}\n\n"

    return transcript, updated_history, ""


def clear_chat():
    return "", [], ""


with gr.Blocks(title="AI Chatbot - 24BAD098") as demo:

    gr.Markdown("# AI Chatbot")
    gr.Markdown("Experiment 10 | Roll Number: 24BAD098")

    conversation = gr.Markdown(
        value="Your conversation will appear here."
    )

    history_state = gr.State([])

    message = gr.Textbox(
        label="Your Message",
        placeholder="Type Hi or ask a question",
        lines=2
    )

    with gr.Row():
        send_button = gr.Button("Send", variant="primary")
        clear_button = gr.Button("Clear Chat")

    send_button.click(
        fn=chat_function,
        inputs=[message, history_state],
        outputs=[conversation, history_state, message]
    )

    message.submit(
        fn=chat_function,
        inputs=[message, history_state],
        outputs=[conversation, history_state, message]
    )

    clear_button.click(
        fn=clear_chat,
        inputs=[],
        outputs=[conversation, history_state, message]
    )

demo.launch(share=True, debug=True)





Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fd37ce3a37bf9ee132.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fd37ce3a37bf9ee132.gradio.live
